In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("data")

# ---------- 0) Loader con fallback di separatore ----------
def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")
dist_km   = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm   = read_csv_auto(DATA/"time_hhmm_matrix.csv")

# ---------- 1) Master list comuni (fonte di verità per etichette) ----------
# municipalities_trentino.csv ha: id, comune, latitude, longitude
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
master_ids   = municipal["id"].tolist()
name_by_id   = dict(zip(municipal["id"], municipal["comune_norm"]))
id_by_name   = dict(zip(municipal["comune_norm"], municipal["id"]))

# ---------- 2) Utilities su tempo e allineamento ----------
def hmm_to_minutes(s):
    s = str(s)
    if ":" not in s:
        return np.nan
    h, m = s.split(":")[:2]
    return int(h)*60 + int(m)

def minutes_to_hmm(minutes):
    if pd.isna(minutes):
        return None
    minutes = int(minutes)
    h, m = divmod(minutes, 60)
    return f"{h}:{m:02d}"

def ensure_matrix_labeled_and_ordered(df, master_names, index_col_name="Unnamed: 0"):
    """
    - Usa la prima colonna come indice (nomi dei comuni)
    - Verifica stessa cardinalità/insieme di etichette su righe/colonne
    - Riordina esattamente come master_names
    - Ritorna (df_riordinato, report_test)
    """
    # Imposta indice = colonna con i nomi riga
    if index_col_name in df.columns:
        df = df.set_index(index_col_name)
    # Rimuovi spazi
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()

    rows = set(df.index)
    cols = set(df.columns)
    master = set(master_names)

    errs = []
    if rows != master:
        miss_r = sorted(list(master - rows))
        extra_r = sorted(list(rows - master))
        errs.append(f"[RIGHE] Mancano nel matrix: {miss_r[:10]} ...; Extra: {extra_r[:10]} ...")
    if cols != master:
        miss_c = sorted(list(master - cols))
        extra_c = sorted(list(cols - master))
        errs.append(f"[COLONNE] Mancano nel matrix: {miss_c[:10]} ...; Extra: {extra_c[:10]} ...")

    if errs:
        raise ValueError("Etichette non coerenti con master list:\n" + "\n".join(errs))

    # Riordina
    df = df.loc[master_names, master_names]
    return df

# ---------- 3) Allinea le matrici distanza/tempo ai nomi master ----------
dist_km_aligned = ensure_matrix_labeled_and_ordered(dist_km.copy(), master_names)
time_hm_aligned = ensure_matrix_labeled_and_ordered(time_hm.copy(), master_names)

# Converte time hh:mm in minuti (stessa etichettatura/ordine)
time_min_aligned = time_hm_aligned.applymap(hmm_to_minutes)

# ---------- 4) Normalizza incoming/outcoming/pop con Codice+Comune ----------
# incoming/outcoming hanno: Codice, Comune, Stesso, Altro, Totale, Lordo, Netto
# pop ha: Codice, popolazione
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)

pop["Codice"] = pop["Codice"].astype(int)

# Controlli di coerenza:
assert incoming["Comune_norm"].isin(municipal["comune_norm"]).all(), "Nomi incoming non allineati alla master list"
assert outcoming["Comune_norm"].isin(municipal["comune_norm"]).all(), "Nomi outcoming non allineati alla master list"
assert set(incoming["Codice"]) == set(municipal["id"]), "Codici incoming diversi da id master"
assert set(outcoming["Codice"]) == set(municipal["id"]), "Codici outcoming diversi da id master"
assert set(pop["Codice"]) == set(municipal["id"]), "Codici pop diversi da id master"

# Join rapido di popolazione per nome e codice (ridondante ma sicuro)
incoming = (incoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

outcoming = (outcoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

# ---------- 5) Calcolo indicatori base (uscite) ----------
# Denominatore = persone attive residenti (stesso + netto)
den_out = (outcoming["Stesso"] + outcoming["Netto"]).replace({0: np.nan})
out_inds = outcoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
out_inds["quota_stesso"] = out_inds["Stesso"] / den_out
out_inds["quota_intra"]  = out_inds["Altro"]  / den_out
out_inds["quota_extra"]  = (out_inds["Netto"] - out_inds["Altro"]) / den_out

# ---------- 6) Indicatori in entrata e saldo ----------
in_inds = incoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
in_inds = in_inds.rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})

# Merge entrata/uscita per comune
indicators = (out_inds
              .merge(in_inds[["Codice","in_Stesso","in_Altro","in_Lordo","in_Netto","in_Totale"]],
                     on="Codice", how="left"))

# Saldi e intensità
indicators["saldo_netto"]   = indicators["in_Totale"] - indicators["Totale"]
indicators["indice_bilancio"] = np.where(indicators["Totale"]>0,
                                         indicators["in_Totale"]/indicators["Totale"], np.nan)
indicators["intensita_pendolarismo"] = (indicators["in_Totale"] + indicators["Totale"]) / indicators["popolazione"]
indicators["autocontenimento_pc"] = indicators["Stesso"] / indicators["popolazione"]
indicators["attrattivita_pc"]     = indicators["in_Totale"] / indicators["popolazione"]

# ---------- 7) Controlli finali su coerenza etichette ----------
def assert_same_order_names(df_matrix, names):
    assert list(df_matrix.index) == names, "Index matrix non coincide con master_names"
    assert list(df_matrix.columns) == names, "Columns matrix non coincide con master_names"

assert_same_order_names(dist_km_aligned, master_names)
assert_same_order_names(time_hm_aligned, master_names)

# ---------- 8) Export "puliti" pronti per le analisi successive ----------
EXPORT = DATA  # cambia se vuoi una cartella dedicata
municipal[["id","comune","latitude","longitude"]].to_csv(EXPORT/"comuni_master.csv", index=False)
dist_km_aligned.to_csv(EXPORT/"distance_km_matrix_aligned.csv")
time_hm_aligned.to_csv(EXPORT/"time_hhmm_matrix_aligned.csv")
time_min_aligned.to_csv(EXPORT/"time_minutes_matrix_aligned.csv")
indicators.to_csv(EXPORT/"indicators_by_comune_2021.csv", index=False)

print("OK ✅  Etichette lette dai file & allineate.\n"
      f"- comuni_master.csv: {len(municipal)} comuni\n"
      f"- distance_km_matrix_aligned.csv: {dist_km_aligned.shape}\n"
      f"- time_minutes_matrix_aligned.csv: {time_min_aligned.shape}\n"
      f"- indicators_by_comune_2021.csv: {indicators.shape}")


OK ✅  Etichette lette dai file & allineate.
- comuni_master.csv: 166 comuni
- distance_km_matrix_aligned.csv: (166, 166)
- time_minutes_matrix_aligned.csv: (166, 166)
- indicators_by_comune_2021.csv: (166, 22)


/tmp/ipykernel_31382/412498493.py:85: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  time_min_aligned = time_hm_aligned.applymap(hmm_to_minutes)


In [11]:
# Phase 1 — Validation & Rankings report for Trentino commuter data (2021)
# This cell builds on the project files already present in /mnt/data.
# It produces CSV exports and a few quick charts.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("data")
EXPORT = DATA / "exports_phase1"
EXPORT.mkdir(exist_ok=True)

def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

# Load base data
municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")
dist_km   = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm   = read_csv_auto(DATA/"time_hhmm_matrix.csv")

# --- Prepare master names ---
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
id_by_name   = dict(zip(municipal["comune_norm"], municipal["id"]))

# Normalize incoming/outcoming/pop
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)
pop["Codice"] = pop["Codice"].astype(int)

# Merge population
incoming = (incoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

outcoming = (outcoming
            .merge(pop.rename(columns={"Codice":"Codice_pop", "popolazione":"popolazione"}),
                   left_on="Codice", right_on="Codice_pop", how="left")
            .drop(columns=["Codice_pop"]))

# Derived indicators
den_out = (outcoming["Stesso"] + outcoming["Netto"]).replace({0: np.nan})

out_inds = outcoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
out_inds["quota_stesso"] = out_inds["Stesso"] / den_out
out_inds["quota_intra"]  = out_inds["Altro"]  / den_out
out_inds["quota_extra"]  = (out_inds["Netto"] - out_inds["Altro"]) / den_out

in_inds = incoming[["Codice","Comune","Comune_norm","Stesso","Altro","Lordo","Netto","Totale","popolazione"]].copy()
in_inds = in_inds.rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})

indicators = (out_inds
              .merge(in_inds[["Codice","in_Stesso","in_Altro","in_Lordo","in_Netto","in_Totale"]],
                     on="Codice", how="left"))

indicators["saldo_netto"]   = indicators["in_Totale"] - indicators["Totale"]
indicators["indice_bilancio"] = np.where(indicators["Totale"]>0,
                                         indicators["in_Totale"]/indicators["Totale"], np.nan)
indicators["intensita_pendolarismo"] = (indicators["in_Totale"] + indicators["Totale"]) / indicators["popolazione"]
indicators["autocontenimento_pc"] = indicators["Stesso"] / indicators["popolazione"]
indicators["attrattivita_pc"]     = indicators["in_Totale"] / indicators["popolazione"]

# --- Provincial consistency check (incoming vs outcoming totals) ---
prov_check = pd.DataFrame({
    "tot_entrate": [int(incoming["Totale"].sum())],
    "tot_uscite":  [int(outcoming["Totale"].sum())],
    "diff":        [int(incoming["Totale"].sum() - outcoming["Totale"].sum())]
})
prov_check.to_csv(EXPORT/"provincial_consistency_check.csv", index=False)

# --- Rankings (top/bottom 10) ---
def top_bottom(df, metric, n=10):
    base = df[["Codice","Comune_norm",metric]].dropna().copy()
    top = base.sort_values(metric, ascending=False).head(n).reset_index(drop=True)
    bottom = base.sort_values(metric, ascending=True).head(n).reset_index(drop=True)
    top["rank"] = np.arange(1, len(top)+1)
    bottom["rank"] = np.arange(1, len(bottom)+1)
    return top, bottom

rank_metrics = ["saldo_netto", "intensita_pendolarismo", "autocontenimento_pc", "attrattivita_pc"]
rank_exports = {}

for m in rank_metrics:
    top, bottom = top_bottom(indicators, m, n=10)
    top.to_csv(EXPORT/f"ranking_top10_{m}.csv", index=False)
    bottom.to_csv(EXPORT/f"ranking_bottom10_{m}.csv", index=False)
    rank_exports[m] = (top, bottom)

# Combine a single wide ranking table (with zscore for comparability)
from scipy.stats import zscore

rank_wide = indicators[["Codice","Comune_norm","saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]].copy()
for c in ["saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]:
    rank_wide[c+"_z"] = zscore(rank_wide[c].fillna(rank_wide[c].mean()))

rank_wide["composite_score"] = rank_wide[[c+"_z" for c in ["saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]]].mean(axis=1)
rank_wide = rank_wide.sort_values("composite_score", ascending=False)
rank_wide.to_csv(EXPORT/"ranking_composite.csv", index=False)

In [14]:

# Display to the user
print("Validation — provincial totals (entrate vs uscite)")
prov_check


Validation — provincial totals (entrate vs uscite)


,tot_entrate,tot_uscite,diff
0,4948,7343,-2395


In [ ]:
print("Ranking (composite score, top)") 
rank_wide.head(20)

Ranking (composite score, top)


,Codice,Comune_norm,saldo_netto,intensita_pendolarismo,autocontenimento_pc,attrattivita_pc,saldo_netto_z,intensita_pendolarismo_z,autocontenimento_pc_z,attrattivita_pc_z,composite_score
16,238,Borgo Chiese,167,0.116896,0.148545,0.101072,3.405064,3.747815,0.837130,7.395376,3.846346
139,183,Storo,151,0.096931,0.178847,0.065136,3.104773,2.961450,1.455330,4.565085,3.021660
68,95,Grigno,115,0.099611,0.156463,0.077745,2.429120,3.067020,0.998654,5.558202,3.013249
118,161,Rovereto,152,0.020020,0.227321,0.011917,3.123541,-0.067778,2.444237,0.373614,1.468404
86,118,Moena / Moena,71,0.037557,0.169575,0.032246,1.603321,0.622932,1.266164,1.974711,1.366782
8,7,Avio,23,0.057946,0.161614,0.031785,0.702449,1.425990,1.103742,1.938404,1.292646
121,164,Sagron Mis,-29,0.184358,0.039106,0.011173,-0.273495,6.404857,-1.395549,0.315055,1.262717
13,17,Bleggio Superiore,56,0.045902,0.099016,0.041311,1.321798,0.951600,-0.173313,2.688711,1.197199
71,102,Lavarone,25,0.037911,0.181129,0.029486,0.739985,0.636866,1.501874,1.757358,1.159021
19,22,Borgo Valsugana,88,0.027217,0.175333,0.019911,1.922379,0.215671,1.383632,1.003250,1.131233


In [ ]:
print("Ranking (composite score, bottom)")
rank_wide.tail(20)


Ranking (composite score, bottom)


,Codice,Comune_norm,saldo_netto,intensita_pendolarismo,autocontenimento_pc,attrattivita_pc,saldo_netto_z,intensita_pendolarismo_z,autocontenimento_pc_z,attrattivita_pc_z,composite_score
90,127,Nogaredo,-13,0.009201,0.062954,0.001453,0.026796,-0.493902,-0.909026,-0.450512,-0.456661
43,54,Cavizzana,-1,0.004167,0.066667,0.000000,0.252013,-0.692184,-0.833284,-0.564932,-0.459597
64,90,Frassilongo / Garait,0,0.000000,0.073529,0.000000,0.270782,-0.856294,-0.693276,-0.564932,-0.460930
58,79,Dro,-38,0.011499,0.078311,0.001983,-0.442408,-0.403399,-0.595729,-0.408788,-0.462581
21,26,Bresimo,-3,0.012000,0.052000,0.000000,0.214477,-0.383659,-1.132500,-0.564932,-0.466653
151,202,Torcegno,-3,0.004418,0.064801,0.000000,0.214477,-0.682275,-0.871342,-0.564932,-0.476018
10,11,Bedollo,-9,0.007478,0.059143,0.000680,0.101868,-0.561767,-0.986766,-0.511391,-0.489514
48,61,Civezzano,-22,0.008321,0.065590,0.001468,-0.142118,-0.528557,-0.855252,-0.449280,-0.493802
161,222,Villa Lagarina,-21,0.009084,0.059694,0.001817,-0.123350,-0.498516,-0.975539,-0.421845,-0.504812
65,91,Garniga Terme,-3,0.007653,0.051020,0.000000,0.214477,-0.554868,-1.152485,-0.564932,-0.514452


In [18]:
# --- Quick charts (matplotlib, one plot each, default colors) ---
def save_bar_top(df, metric, title, fname):
    top = df.sort_values(metric, ascending=False).head(10)
    plt.figure(figsize=(10,4))
    plt.bar(top["Comune_norm"], top[metric])
    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(EXPORT/fname)
    plt.close()

save_bar_top(indicators, "saldo_netto", "Top 10 comuni per saldo netto (entrate - uscite)", "chart_top10_saldo_netto.png")
save_bar_top(indicators, "intensita_pendolarismo", "Top 10 intensità di pendolarismo (entrate+uscite / popolazione)", "chart_top10_intensita.png")
save_bar_top(indicators, "attrattivita_pc", "Top 10 attrattività pro capite (entrate/pop)", "chart_top10_attrattivita.png")
save_bar_top(indicators, "autocontenimento_pc", "Top 10 autocontenimento pro capite (stesso/pop)", "chart_top10_autocontenimento.png")

# Export a clean indicators table
indicators_sorted = indicators.sort_values("Comune_norm")
indicators_sorted.to_csv(EXPORT/"indicators_by_comune_2021_phase1.csv", index=False)

# Final message to the notebook output
print("Phase 1 complete.\n"
      f"Exports in: {EXPORT}\n"
      "- provincial_consistency_check.csv\n"
      "- ranking_top10_*.csv / ranking_bottom10_*.csv\n"
      "- ranking_composite.csv\n"
      "- indicators_by_comune_2021_phase1.csv\n"
      "- chart_*.png")


Phase 1 complete.
Exports in: data/exports_phase1
- provincial_consistency_check.csv
- ranking_top10_*.csv / ranking_bottom10_*.csv
- ranking_composite.csv
- indicators_by_comune_2021_phase1.csv
- chart_*.png


In [1]:
# Re-run Phase 2 — Spatial accessibility metrics and point maps

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("data")
EXPORT = DATA / "exports_phase2"
EXPORT.mkdir(exist_ok=True)

def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

# Load base data
municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
indicators = read_csv_auto(DATA/"exports_phase1/indicators_by_comune_2021_phase1.csv")
dist_km = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm = read_csv_auto(DATA/"time_hhmm_matrix.csv")
pop = read_csv_auto(DATA/"trentino_population_2021.csv")

# Master order & coords
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
coords = municipal.set_index("comune_norm")[["latitude","longitude"]]

# Align matrices
def ensure_matrix(df, master_names, index_col_name="Unnamed: 0"):
    if index_col_name in df.columns:
        df = df.set_index(index_col_name)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    df = df.loc[master_names, master_names]
    return df

dist_km_al = ensure_matrix(dist_km.copy(), master_names)
time_hm_al = ensure_matrix(time_hm.copy(), master_names)

# hh:mm -> minutes
def hmm_to_minutes(s):
    s = str(s)
    if ":" not in s:
        return np.nan
    h, m = s.split(":")[:2]
    return int(h)*60 + int(m)

time_min_al = time_hm_al.applymap(hmm_to_minutes)

# Population vector aligned
pop_vec = (pop.rename(columns={"Codice":"id"})
              .merge(municipal[["id","comune_norm"]], on="id", how="right")
              .set_index("comune_norm")["popolazione"]
              .astype(float)
          )

# --- Accessibility Potentials ---
beta = 1.5
dist_np = dist_km_al.to_numpy().astype(float)
time_np = time_min_al.to_numpy().astype(float)
pop_np = pop_vec.to_numpy().astype(float)

# Avoid diagonal
dist_np_z = dist_np.copy()
np.fill_diagonal(dist_np_z, np.nan)
time_np_z = time_np.copy()
np.fill_diagonal(time_np_z, np.nan)

with np.errstate(divide='ignore', invalid='ignore'):
    A_km = np.nansum(pop_np / np.power(dist_np_z, beta), axis=1)
    A_t  = np.nansum(pop_np / (1.0 + time_np_z), axis=1)

access_df = pd.DataFrame({
    "Comune_norm": master_names,
    "A_km_beta1_5": A_km,
    "A_time": A_t
})

# Neighbor counts
km_thresholds = [10, 20, 30]
min_thresholds = [15, 30, 45]
neighbor_counts = {"Comune_norm": master_names}
for thr in km_thresholds:
    cnt = np.sum((dist_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
    neighbor_counts[f"neighbors_km_le_{thr}"] = cnt
for thr in min_thresholds:
    cnt = np.sum((time_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
    neighbor_counts[f"neighbors_min_le_{thr}"] = cnt
neighbors_df = pd.DataFrame(neighbor_counts)

# Population reach radii (25% / 50% of provincial pop)
total_pop = np.nansum(pop_np)
targets = [0.25*total_pop, 0.50*total_pop]

def reach_thresholds(matrix, pop_np, targets):
    n = matrix.shape[0]
    out = np.full((n, len(targets)), np.nan)
    for i in range(n):
        vals = matrix[i, :].copy()
        vals[i] = np.nan
        order = np.argsort(vals)
        sorted_d = vals[order]
        sorted_p = pop_np[order]
        # NaNs will be at the end; treat as zero population to keep cumulative stable
        sp = np.nan_to_num(sorted_p, nan=0.0)
        cum = np.cumsum(sp)
        for k, tgt in enumerate(targets):
            idx = np.searchsorted(cum, tgt, side="left")
            if idx < len(sorted_d):
                out[i, k] = sorted_d[idx]
            else:
                out[i, k] = np.nan
    return out

r_km = reach_thresholds(dist_np, pop_np, targets)
r_min = reach_thresholds(time_np, pop_np, targets)

radii_df = pd.DataFrame({
    "Comune_norm": master_names,
    "R25_km": r_km[:,0], "R50_km": r_km[:,1],
    "T25_min": r_min[:,0], "T50_min": r_min[:,1],
})

# Merge with indicators
ind_small = indicators[["Codice","Comune_norm","saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc"]].copy()
acc_all = (ind_small
           .merge(access_df, on="Comune_norm", how="left")
           .merge(neighbors_df, on="Comune_norm", how="left")
           .merge(radii_df, on="Comune_norm", how="left")
          )

# Save CSVs
access_df.to_csv(EXPORT/"accessibility_potential.csv", index=False)
neighbors_df.to_csv(EXPORT/"neighbors_counts.csv", index=False)
radii_df.to_csv(EXPORT/"population_reach_radii.csv", index=False)
acc_all.to_csv(EXPORT/"accessibility_metrics_all.csv", index=False)

# Point maps (static)
coords2 = coords.reindex(master_names)

def point_map(metric_series, title, fname):
    x = coords2["longitude"].values
    y = coords2["latitude"].values
    # normalize to sizes
    mi = np.nanmin(metric_series)
    ma = np.nanmax(metric_series)
    s = (metric_series - mi) / (ma - mi + 1e-9)
    s = 50 + 450*s
    plt.figure(figsize=(6,7))
    plt.scatter(x, y, s=s)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.savefig(EXPORT/fname, dpi=150)
    plt.close()

point_map(acc_all["attrattivita_pc"].to_numpy(), "Attrattività pro capite (dimensione punto)", "map_attrattivita_pc.png")
point_map(acc_all["intensita_pendolarismo"].to_numpy(), "Intensità di pendolarismo (dimensione punto)", "map_intensita_pendolarismo.png")
point_map(acc_all["A_km_beta1_5"].to_numpy(), "Accessibilità potenziale (km, β=1.5)", "map_accessibilita_potenziale.png")

# Show a preview
preview = acc_all.sort_values("A_km_beta1_5", ascending=False).head(20)
print("Phase 2 — Top 20 comuni per accessibilità potenziale (km, β=1.5)")
preview

print("Phase 2 complete.\n"
      f"Exports in: {EXPORT}\n"
      "- accessibility_potential.csv\n"
      "- neighbors_counts.csv\n"
      "- population_reach_radii.csv\n"
      "- accessibility_metrics_all.csv\n"
      "- map_attrattivita_pc.png\n"
      "- map_intensita_pendolarismo.png\n"
      "- map_accessibilita_potenziale.png")


/tmp/ipykernel_54667/1230458786.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  time_min_al = time_hm_al.applymap(hmm_to_minutes)


Phase 2 — Top 20 comuni per accessibilità potenziale (km, β=1.5)
Phase 2 complete.
Exports in: data/exports_phase2
- accessibility_potential.csv
- neighbors_counts.csv
- population_reach_radii.csv
- accessibility_metrics_all.csv
- map_attrattivita_pc.png
- map_intensita_pendolarismo.png
- map_accessibilita_potenziale.png


In [3]:
EXP2 = DATA / "exports_phase2"
EXP2.mkdir(exist_ok=True)
PROFILES = EXP2 / "profiles"
PROFILES.mkdir(exist_ok=True)

def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

# Load data
municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
dist_km = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm = read_csv_auto(DATA/"time_hhmm_matrix.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")

# Master order & coords
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
coords = municipal.set_index("comune_norm")[["latitude","longitude"]]
id_map = municipal.set_index("comune_norm")["id"].to_dict()

# Align matrices
def ensure_matrix(df, master_names, index_col_name="Unnamed: 0"):
    if index_col_name in df.columns:
        df = df.set_index(index_col_name)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    df = df.loc[master_names, master_names]
    return df

dist_km_al = ensure_matrix(dist_km.copy(), master_names)
time_hm_al = ensure_matrix(time_hm.copy(), master_names)

# hh:mm -> minutes
def hmm_to_minutes(s):
    s = str(s)
    if ":" not in s:
        return np.nan
    h, m = s.split(":")[:2]
    return int(h)*60 + int(m)

time_min_al = time_hm_al.applymap(hmm_to_minutes)

# Population vector aligned
pop_vec = (pop.rename(columns={"Codice":"id"})
              .merge(municipal[["id","comune_norm"]], on="id", how="right")
              .set_index("comune_norm")["popolazione"]
              .astype(float)
          )

# Prepare incoming/outcoming normalized
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)

incoming_min = incoming[["Codice","Comune_norm","Stesso","Altro","Lordo","Netto","Totale"]].rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})
outcoming_min = outcoming[["Codice","Comune_norm","Stesso","Altro","Lordo","Netto","Totale"]].rename(columns={
    "Totale":"out_Totale"
})

# Merge (columns from outcoming_min keep names like 'Stesso','Altro', etc.)
ind = (outcoming_min
       .merge(incoming_min, on="Codice", how="left", suffixes=("", "_in")))

# Indicators
# Use Comune_norm from outcoming side for alignment with population
pop_aligned = pop_vec.reindex(ind["Comune_norm"])
ind["saldo_netto"] = ind["in_Totale"] - ind["out_Totale"]
ind["intensita_pendolarismo"] = (ind["in_Totale"] + ind["out_Totale"]) / pop_aligned.values
ind["autocontenimento_pc"] = ind["Stesso"] / pop_aligned.values
ind["attrattivita_pc"] = ind["in_Totale"] / pop_aligned.values

# Accessibility metrics
beta = 1.5
dist_np = dist_km_al.to_numpy().astype(float)
time_np = time_min_al.to_numpy().astype(float)
pop_np = pop_vec.reindex(master_names).to_numpy().astype(float)

dist_np_z = dist_np.copy(); np.fill_diagonal(dist_np_z, np.nan)
time_np_z = time_np.copy(); np.fill_diagonal(time_np_z, np.nan)

with np.errstate(divide='ignore', invalid='ignore'):
    A_km = np.nansum(pop_np / np.power(dist_np_z, beta), axis=1)
    A_t  = np.nansum(pop_np / (1.0 + time_np_z), axis=1)

# Neighbor counts
neighbors = {"Comune_norm": master_names}
for thr in [10,20,30]:
    neighbors[f"neighbors_km_le_{thr}"] = np.sum((dist_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
for thr in [15,30,45]:
    neighbors[f"neighbors_min_le_{thr}"] = np.sum((time_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
neighbors_df = pd.DataFrame(neighbors)

# Reach radii
total_pop = np.nansum(pop_np)
targets = [0.25*total_pop, 0.50*total_pop]

def reach_thresholds(matrix, pop_np, targets):
    n = matrix.shape[0]
    out = np.full((n, len(targets)), np.nan)
    for i in range(n):
        vals = matrix[i, :].copy()
        vals[i] = np.nan
        order = np.argsort(vals)
        sorted_d = vals[order]
        sorted_p = pop_np[order]
        sp = np.nan_to_num(sorted_p, nan=0.0)
        cum = np.cumsum(sp)
        for k, tgt in enumerate(targets):
            idx = np.searchsorted(cum, tgt, side="left")
            out[i, k] = sorted_d[idx] if idx < len(sorted_d) else np.nan
    return out

r_km = reach_thresholds(dist_np, pop_np, targets)
r_min = reach_thresholds(time_np, pop_np, targets)

acc = pd.DataFrame({
    "Comune_norm": master_names,
    "A_km_beta1_5": A_km,
    "A_time": A_t,
    "R25_km": r_km[:,0], "R50_km": r_km[:,1],
    "T25_min": r_min[:,0], "T50_min": r_min[:,1],
}).merge(neighbors_df, on="Comune_norm", how="left")

# Merge indicators with accessibility (by Comune_norm). Attach Codice from municipal
acc_all = (ind[["Codice","Comune_norm","saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc","out_Totale","in_Totale"]]
           .merge(acc, on="Comune_norm", how="left"))
acc_all = acc_all.merge(municipal[["id","comune_norm"]], left_on=["Codice","Comune_norm"], right_on=["id","comune_norm"], how="left")

# Map inverse R50_km
acc_all["inv_R50_km"] = 1.0 / acc_all["R50_km"]
xy = coords.reindex(acc_all["Comune_norm"])
sizes = 50 + 450 * ((acc_all["inv_R50_km"] - np.nanmin(acc_all["inv_R50_km"])) / (np.nanmax(acc_all["inv_R50_km"]) - np.nanmin(acc_all["inv_R50_km"]) + 1e-9))
plt.figure(figsize=(6,7))
plt.scatter(xy["longitude"], xy["latitude"], s=sizes)
plt.title("Accessibilità (R50 km → inverso): più grande = più popolazione raggiunta a corto raggio")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
map_path = EXP2/"map_inverse_R50km.png"
plt.savefig(map_path, dpi=150)
plt.close()

# Ranks & percentiles
def add_rank_pct(df, col, ascending=False):
    df[col+"_rank"] = df[col].rank(ascending=ascending, method="min")
    df[col+"_pct"]  = df[col].rank(pct=True, ascending=ascending)*100
    return df

prof = acc_all.copy()
prof = add_rank_pct(prof, "saldo_netto", ascending=False)
prof = add_rank_pct(prof, "intensita_pendolarismo", ascending=False)
prof = add_rank_pct(prof, "autocontenimento_pc", ascending=False)
prof = add_rank_pct(prof, "attrattivita_pc", ascending=False)
prof = add_rank_pct(prof, "A_km_beta1_5", ascending=False)
prof = add_rank_pct(prof, "R50_km", ascending=True)
prof = add_rank_pct(prof, "T50_min", ascending=True)
prof = add_rank_pct(prof, "neighbors_km_le_20", ascending=False)
prof = add_rank_pct(prof, "neighbors_min_le_30", ascending=False)

profiles_csv = EXP2/"profiles_all_metrics.csv"
prof.to_csv(profiles_csv, index=False)

# Markdown profiles
def fmt_num(x):
    if pd.isna(x):
        return "—"
    try:
        xf = float(x)
        if abs(xf) >= 1000 and float(xf).is_integer():
            return f"{int(xf):,}".replace(",", ".")
        if abs(xf) >= 100:
            return f"{xf:.1f}"
        return f"{xf:.3f}"
    except Exception:
        return str(x)

def write_profile_row(row, n_total):
    name = row["Comune_norm"]
    cid = id_map.get(name, None)
    path = PROFILES / (f"{cid:03d}_{name.replace('/', '-')}.md" if cid is not None else f"{name.replace('/', '-')}.md")
    L = []
    L.append(f"# {name} — Mobility Profile (2021)\n")
    if cid is not None:
        L.append(f"**ID:** {cid}\n")
    L.append("## Population & Flows\n")
    L.append(f"- **Entrate Totali**: {fmt_num(row.get('in_Totale'))}")
    L.append(f"- **Uscite Totali**: {fmt_num(row.get('out_Totale'))}")
    L.append(f"- **Saldo netto**: {fmt_num(row.get('saldo_netto'))}  \n")
    L.append("## Indicatori chiave\n")
    L.append(f"- **Intensità pendolarismo**: {fmt_num(row.get('intensita_pendolarismo'))} (rank {fmt_num(row.get('intensita_pendolarismo_rank'))}/{n_total})")
    L.append(f"- **Autocontenimento pc**: {fmt_num(row.get('autocontenimento_pc'))} (rank {fmt_num(row.get('autocontenimento_pc_rank'))}/{n_total})")
    L.append(f"- **Attrattività pc**: {fmt_num(row.get('attrattivita_pc'))} (rank {fmt_num(row.get('attrattivita_pc_rank'))}/{n_total})")
    L.append(f"- **A_km (β=1.5)**: {fmt_num(row.get('A_km_beta1_5'))} (rank {fmt_num(row.get('A_km_beta1_5_rank'))}/{n_total})  \n")
    L.append("## Raggi di popolazione\n")
    L.append(f"- **R50_km**: {fmt_num(row.get('R50_km'))} km (rank {fmt_num(row.get('R50_km_rank'))}/{n_total})")
    L.append(f"- **T50_min**: {fmt_num(row.get('T50_min'))} min (rank {fmt_num(row.get('T50_min_rank'))}/{n_total})")
    L.append("## Vicinato (connettività)\n")
    L.append(f"- Entro **20 km**: {fmt_num(row.get('neighbors_km_le_20'))} (rank {fmt_num(row.get('neighbors_km_le_20_rank'))}/{n_total})")
    L.append(f"- Entro **30 min**: {fmt_num(row.get('neighbors_min_le_30'))} (rank {fmt_num(row.get('neighbors_min_le_30_rank'))}/{n_total})\n")
    L.append("---\n")
    L.append("_Generated by tracetn — Commuter Mobility Analysis (2021)._")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(L))
    return path

paths = []
n_total = len(prof)
for _, r in prof.iterrows():
    paths.append(str(write_profile_row(r, n_total)))

index_df = pd.DataFrame({"Comune_norm": prof["Comune_norm"], "profile_path": paths})
index_csv = EXP2/"profiles_index.csv"
index_df.to_csv(index_csv, index=False)



print("OK:")
print(f"- Map: {map_path}")
print(f"- All metrics CSV: {profiles_csv}")
print(f"- Profiles index: {index_csv}")
print(f"- Profiles folder: {PROFILES}")
# Show sample
print("Profiles index (sample)")
index_df.head(12)


OK:
- Map: data/exports_phase2/map_inverse_R50km.png
- All metrics CSV: data/exports_phase2/profiles_all_metrics.csv
- Profiles index: data/exports_phase2/profiles_index.csv
- Profiles folder: data/exports_phase2/profiles
Profiles index (sample)


/tmp/ipykernel_54667/4169796036.py:47: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  time_min_al = time_hm_al.applymap(hmm_to_minutes)


,Comune_norm,profile_path
0,Ala,data/exports_phase2/profiles/001_Ala.md
1,Albiano,data/exports_phase2/profiles/002_Albiano.md
2,Aldeno,data/exports_phase2/profiles/003_Aldeno.md
3,Altavalle,data/exports_phase2/profiles/235_Altavalle.md
4,Altopiano della Vigolana,data/exports_phase2/profiles/236_Altopiano del...
5,Amblar-Don,data/exports_phase2/profiles/237_Amblar-Don.md
6,Andalo,data/exports_phase2/profiles/005_Andalo.md
7,Arco,data/exports_phase2/profiles/006_Arco.md
8,Avio,data/exports_phase2/profiles/007_Avio.md
9,Baselga di Pinè,data/exports_phase2/profiles/009_Baselga di Pi...


In [8]:
# Retry Kepler HTML build with escaped braces; enrich profiles with mean/quartiles (re-run safely).



import json

# Load metrics and coordinates
municipal = pd.read_csv(DATA/"municipalities_trentino.csv")
metrics   = pd.read_csv(EXP2/"profiles_all_metrics.csv")

municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
coords = municipal[["id","comune_norm","latitude","longitude"]]

kepler_df = (metrics.merge(coords, left_on="Codice", right_on="id", how="left")
             .rename(columns={"comune_norm":"Comune", "latitude":"lat", "longitude":"lon"}))

# Save CSV
kepler_csv_path = EXP2 / "kepler_points.csv"
kepler_df.to_csv(kepler_csv_path, index=False)

# Prepare embedded data
data_fields = list(kepler_df.columns)
fields_spec = [{"name": c, "type": "real" if pd.api.types.is_numeric_dtype(kepler_df[c]) else "string"} for c in data_fields]
rows = kepler_df.where(pd.notnull(kepler_df), None).values.tolist()

config = {
  "version": "v1",
  "config": {
    "visState": {
      "filters": [],
      "layers": [{
        "id": "municipal_points",
        "type": "point",
        "config": {
          "dataId": "Comuni",
          "label": "Comuni Trentino",
          "color": [18, 147, 154],
          "columns": {"lat": "lat", "lng": "lon"},
          "isVisible": True,
          "visConfig": {"radius": 8, "opacity": 0.8, "outline": False}
        },
        "visualChannels": {
          "sizeField": {"name": "A_km_beta1_5", "type": "real"},
          "sizeScale": "sqrt"
        }
      }],
      "interactionConfig": {
        "tooltip": {
          "fieldsToShow": {
            "Comuni": [
              {"name":"Comune"},
              {"name":"saldo_netto"},
              {"name":"intensita_pendolarismo"},
              {"name":"attrattivita_pc"},
              {"name":"A_km_beta1_5"},
              {"name":"R50_km"},
              {"name":"T50_min"}
            ]
          },
          "enabled": True
        }
      }
    },
    "mapState": {"latitude":46.1,"longitude":11.1,"zoom":8},
    "mapStyle": {"styleType":"light"}
  }
}

html_template = """
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>tracetn — Kepler Map (Trentino 2021)</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <style>html,body,#app{{margin:0;height:100%;}}</style>
  <script src="https://unpkg.com/react@17/umd/react.production.min.js"></script>
  <script src="https://unpkg.com/react-dom@17/umd/react-dom.production.min.js"></script>
  <script src="https://unpkg.com/redux@4.0.5/dist/redux.min.js"></script>
  <script src="https://unpkg.com/react-redux@7.2.1/dist/react-redux.min.js"></script>
  <script src="https://unpkg.com/styled-components/dist/styled-components.min.js"></script>
  <script src="https://unpkg.com/kepler.gl/umd/keplergl.min.js"></script>
</head>
<body>
<div id="app"></div>
<script>
  const {{KeplerGl, addDataToMap, KeplerGlFactory}} = window.keplergl;
  const container = document.getElementById('app');

  const dataset = {{
    info: {{ id: "Comuni", label: "Comuni" }},
    data: {{
      fields: <<FIELDS>>,
      rows: <<ROWS>>
    }}
  }};

  const keplerConfig = <<CONFIG>>;

  const App = () => React.createElement(KeplerGl, {{
    id: "map",
    width: window.innerWidth,
    height: window.innerHeight
  }});

  ReactDOM.render(React.createElement(App), container);

  setTimeout(() => {{
    const action = window.keplergl.addDataToMap({{
      datasets: dataset,
      options: {{centerMap: true}},
      config: keplerConfig.config
    }});
    window.keplergl.store.dispatch(action);
  }}, 500);
</script>
</body>
</html>
"""

html_filled = (html_template
               .replace("<<FIELDS>>", json.dumps(fields_spec))
               .replace("<<ROWS>>", json.dumps(rows))
               .replace("<<CONFIG>>", json.dumps(config)))

kepler_html_path = EXP2 / "tracetn_kepler_map.html"
with open(kepler_html_path, "w", encoding="utf-8") as f:
    f.write(html_filled)

# ---------------- Enrich profiles with mean & quartiles ----------------
metric_cols = [
    ("saldo_netto", "higher"),
    ("intensita_pendolarismo", "higher"),
    ("autocontenimento_pc", "higher"),
    ("attrattivita_pc", "higher"),
    ("A_km_beta1_5", "higher"),
    ("R50_km", "lower"),
    ("T50_min", "lower"),
    ("neighbors_km_le_20", "higher"),
    ("neighbors_min_le_30", "higher"),
]

summary_rows = []
for col, direction in metric_cols:
    s = metrics[col].astype(float)
    summary_rows.append({
        "metric": col,
        "direction": direction,
        "mean": float(np.nanmean(s)),
        "q1": float(np.nanpercentile(s, 25)),
        "median": float(np.nanpercentile(s, 50)),
        "q3": float(np.nanpercentile(s, 75))
    })
summary_df = pd.DataFrame(summary_rows)
summary_csv = EXP2 / "metrics_summary_quartiles.csv"
summary_df.to_csv(summary_csv, index=False)

def quartile_bucket(x, q1, q2, q3, direction):
    if pd.isna(x): return "n.d."
    if direction == "higher":
        return "Q1 (basso)" if x < q1 else ("Q2" if x < q2 else ("Q3" if x < q3 else "Q4 (alto)"))
    else:
        return "Q4 (migliore)" if x < q1 else ("Q3" if x < q2 else ("Q2" if x < q3 else "Q1 (peggiore)"))

def fmt_num(x):
    if pd.isna(x): return "—"
    try:
        xf = float(x)
        if abs(xf) >= 1000 and float(xf).is_integer():
            return f"{int(xf):,}".replace(",", ".")
        if abs(xf) >= 100:
            return f"{xf:.1f}"
        return f"{xf:.3f}"
    except Exception:
        return str(x)

# Ensure profiles exist; if not, create a minimal base
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
id_by_name = municipal.set_index("comune_norm")["id"].to_dict()
metrics_by_name = metrics.set_index("Comune_norm")

created = 0
updated = 0

for name, row in metrics_by_name.iterrows():
    cid = id_by_name.get(name, None)
    md_path = PROFILES / (f"{cid:03d}_{name.replace('/', '-')}.md" if cid is not None else f"{name.replace('/', '-')}.md")

    if not md_path.exists():
        base = [
            f"# {name} — Mobility Profile (2021)",
            f"**ID:** {cid}" if cid is not None else "",
            "",
            "## Indicatori chiave (estratto)",
            f"- Saldo netto: {fmt_num(row['saldo_netto'])}",
            f"- Intensità pendolarismo: {fmt_num(row['intensita_pendolarismo'])}",
            f"- Autocontenimento pc: {fmt_num(row['autocontenimento_pc'])}",
            f"- Attrattività pc: {fmt_num(row['attrattivita_pc'])}",
            ""
        ]
        with open(md_path, "w", encoding="utf-8") as f:
            f.write("\n".join([l for l in base if l is not None]))
        created += 1

    comp_lines = []
    comp_lines.append("\n## Confronti con media provinciale e quartili\n")
    comp_lines.append("| Metrica | Valore | Media | Q1 | Mediana | Q3 | Posizione |")
    comp_lines.append("|---|---:|---:|---:|---:|---:|---|")

    for col, direction in metric_cols:
        s = summary_df[summary_df["metric"]==col].iloc[0]
        x = row[col]
        bucket = quartile_bucket(x, s["q1"], s["median"], s["q3"], direction)
        comp_lines.append(f"| {col} | {fmt_num(x)} | {fmt_num(s['mean'])} | {fmt_num(s['q1'])} | {fmt_num(s['median'])} | {fmt_num(s['q3'])} | {bucket} |")

    with open(md_path, "a", encoding="utf-8") as f:
        f.write("\n" + "\n".join(comp_lines) + "\n")
    updated += 1



print("Kepler files:")
print(f"- CSV: {kepler_csv_path}")
print(f"- HTML (open with internet active): {kepler_html_path}")
print("Profiles updated with mean/quartile comparisons.")
print(f"Profiles created: {created}, updated: {updated}")


Kepler files:
- CSV: data/exports_phase2/kepler_points.csv
- HTML (open with internet active): data/exports_phase2/tracetn_kepler_map.html
Profiles updated with mean/quartile comparisons.
Profiles created: 0, updated: 166


In [9]:
# Show a couple of outputs
print("Kepler dataset (sample)")
kepler_df.head(12)


Kepler dataset (sample)


,Codice,Comune_norm,saldo_netto,intensita_pendolarismo,autocontenimento_pc,attrattivita_pc,out_Totale,in_Totale,A_km_beta1_5,A_time,...,T50_min_rank,T50_min_pct,neighbors_km_le_20_rank,neighbors_km_le_20_pct,neighbors_min_le_30_rank,neighbors_min_le_30_pct,id_y,comune_norm_y,lat,lon
0,1,Ala,-47,0.022621,0.136751,0.008639,123,76,2777.396079,15333.765239,...,48.0,31.024096,123.0,79.216867,116.0,72.289157,1,Ala,45.762550,11.005162
1,2,Albiano,-20,0.013360,0.137609,0.000000,20,0,4597.508661,19025.926405,...,29.0,18.373494,77.0,48.192771,29.0,18.373494,2,Albiano,46.144377,11.194003
2,3,Aldeno,-28,0.011292,0.092221,0.001255,32,4,5898.445126,23814.499298,...,1.0,1.807229,50.0,34.337349,15.0,9.638554,3,Aldeno,45.980792,11.091534
3,235,Altavalle,-3,0.008020,0.095620,0.003085,8,5,4738.768504,17108.244341,...,40.0,25.602410,50.0,34.337349,64.0,39.759036,235,Altavalle,46.180672,11.233162
4,236,Altopiano della Vigolana,-23,0.008811,0.079107,0.002154,34,11,5747.196365,20678.699153,...,14.0,8.433735,65.0,42.469880,11.0,6.626506,236,Altopiano della Vigolana,46.005422,11.199894
5,237,Amblar-Don,-14,0.032907,0.120658,0.003656,16,2,2809.748550,11421.753291,...,113.0,68.975904,77.0,48.192771,103.0,62.951807,237,Amblar-Don,46.395047,11.146679
6,5,Andalo,7,0.005800,0.224524,0.005800,0,7,2309.442119,14038.943388,...,66.0,40.060241,116.0,71.686747,75.0,46.686747,5,Andalo,46.164038,11.002529
7,6,Arco,39,0.018540,0.225416,0.010369,145,184,4664.986973,16417.887199,...,48.0,31.024096,109.0,67.469880,75.0,46.686747,6,Arco,45.918023,10.886020
8,7,Avio,23,0.057946,0.161614,0.031785,107,130,2273.880642,14727.407551,...,48.0,31.024096,161.0,98.493976,138.0,84.036145,7,Avio,45.735276,10.939340
9,9,Baselga di Pinè,10,0.011797,0.118954,0.006882,25,35,4431.316032,18242.263620,...,35.0,22.289157,15.0,11.445783,58.0,36.445783,9,Baselga di Pinè,46.129862,11.244047


In [10]:
print("Metriche — media e quartili")
summary_df

Metriche — media e quartili


,metric,direction,mean,q1,median,q3
0,saldo_netto,higher,-14.427711,-22.000000,-6.000000,0.000000
1,intensita_pendolarismo,higher,0.021741,0.008260,0.012991,0.023631
2,autocontenimento_pc,higher,0.107512,0.073693,0.094553,0.133113
3,attrattivita_pc,higher,0.007173,0.000830,0.003072,0.009061
4,A_km_beta1_5,higher,3566.948372,2280.622684,3016.362409,4415.889111
5,R50_km,lower,52.356133,40.211750,50.766500,62.546750
6,T50_min,lower,49.891566,37.000000,47.500000,59.750000
7,neighbors_km_le_20,higher,13.843373,9.000000,14.500000,18.000000
8,neighbors_min_le_30,higher,25.819277,17.250000,24.000000,34.000000


In [13]:
# Generate a single Markdown file with a profile section for every municipality.
# Each section includes:
# - Anchor + subtitle with the municipality name
# - Natural-language description based on indicators
# - A table with indicator values and differences from provincial means
# - A list of similar municipalities (links to anchors)
#
# Inputs: base CSVs in /mnt/data; outputs: /mnt/data/municipality_profiles.md


import math
import unicodedata

DATA = Path("data")
DOCS = Path("documentation")
OUT_MD = DOCS / "profili_comuni.md"

# ---------- Helpers ----------
def read_csv_auto(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep=";")

def slugify(text):
    # Simple slug/anchor id: lowercase, ascii, replace spaces with '-', keep alnum and '-'
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii","ignore").decode("ascii")
    text = text.lower().strip().replace(" ", "-")
    text = "".join(ch for ch in text if ch.isalnum() or ch in "-_")
    return text

def fmt_num(x, digits=3):
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "—"
    try:
        xf = float(x)
        if abs(xf) >= 1000 and float(xf).is_integer():
            return f"{int(xf):,}".replace(",", ".")
        if abs(xf) >= 100:
            return f"{xf:.1f}"
        return f"{xf:.{digits}f}"
    except Exception:
        return str(x)

def quantile_label(val, q1, q2, q3, higher_is_better=True):
    if val is None or (isinstance(val, float) and (math.isnan(val) or math.isinf(val))):
        return "n.d."
    if higher_is_better:
        if val < q1: return "low (Q1)"
        if val < q2: return "mid-low (Q2)"
        if val < q3: return "mid-high (Q3)"
        return "high (Q4)"
    else:
        # lower is better -> reverse
        if val < q1: return "high (Q4)"
        if val < q2: return "mid-high (Q3)"
        if val < q3: return "mid-low (Q2)"
        return "low (Q1)"

# ---------- Load base data ----------
municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")
dist_km   = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm   = read_csv_auto(DATA/"time_hhmm_matrix.csv")

# Normalize names / ids
municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()
id_by_name = dict(zip(municipal["comune_norm"], municipal["id"]))

# Align matrices to master names
def ensure_matrix(df, master_names, index_col_name="Unnamed: 0"):
    if index_col_name in df.columns:
        df = df.set_index(index_col_name)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    df = df.loc[master_names, master_names]
    return df

dist_km = ensure_matrix(dist_km, master_names)
time_hm = ensure_matrix(time_hm, master_names)

# Convert hh:mm -> minutes
def hmm_to_minutes(s):
    s = str(s)
    if ":" not in s:
        return np.nan
    h, m = s.split(":")[:2]
    return int(h)*60 + int(m)

time_min = time_hm.applymap(hmm_to_minutes)

# Prepare incoming/outgoing and population
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)
pop["Codice"] = pop["Codice"].astype(int)

incoming_min = incoming[["Codice","Comune_norm","Stesso","Altro","Lordo","Netto","Totale"]].rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})
outcoming_min = outcoming[["Codice","Comune_norm","Stesso","Altro","Lordo","Netto","Totale"]].rename(columns={
    "Totale":"out_Totale"
})

ind = (outcoming_min
       .merge(incoming_min, on="Codice", how="left", suffixes=("_out","_in"))
       .merge(pop[["Codice","popolazione"]], on="Codice", how="left"))

# Align by name
ind = ind.rename(columns={"Comune_norm_out":"Comune_norm"})
# Compute core indicators
den_out = (ind["Stesso"] + ind["Netto"]).replace({0: np.nan})
ind["saldo_netto"] = ind["in_Totale"] - ind["out_Totale"]
ind["intensita_pendolarismo"] = (ind["in_Totale"] + ind["out_Totale"]) / ind["popolazione"]
ind["autocontenimento_pc"] = ind["Stesso"] / ind["popolazione"]
ind["attrattivita_pc"] = ind["in_Totale"] / ind["popolazione"]
ind["indice_bilancio"] = np.where(ind["out_Totale"]>0, ind["in_Totale"]/ind["out_Totale"], np.nan)

# Accessibility metrics
pop_vec = (pop.rename(columns={"Codice":"id"})
              .merge(municipal[["id","comune_norm"]], on="id", how="right")
              .set_index("comune_norm")["popolazione"]
              .astype(float)
          )
beta = 1.5
dist_np = dist_km.to_numpy().astype(float)
time_np = time_min.to_numpy().astype(float)
pop_np = pop_vec.reindex(master_names).to_numpy().astype(float)

dist_np_z = dist_np.copy(); np.fill_diagonal(dist_np_z, np.nan)
time_np_z = time_np.copy(); np.fill_diagonal(time_np_z, np.nan)

with np.errstate(divide='ignore', invalid='ignore'):
    A_km = np.nansum(pop_np / np.power(dist_np_z, beta), axis=1)
    A_t  = np.nansum(pop_np / (1.0 + time_np_z), axis=1)

def reach_thresholds(matrix, pop_np, targets):
    n = matrix.shape[0]
    out = np.full((n, len(targets)), np.nan)
    for i in range(n):
        vals = matrix[i, :].copy()
        vals[i] = np.nan
        order = np.argsort(vals)
        sorted_d = vals[order]
        sorted_p = pop_np[order]
        sp = np.nan_to_num(sorted_p, nan=0.0)
        cum = np.cumsum(sp)
        for k, tgt in enumerate(targets):
            idx = np.searchsorted(cum, tgt, side="left")
            out[i, k] = sorted_d[idx] if idx < len(sorted_d) else np.nan
    return out

total_pop = np.nansum(pop_np)
targets = [0.25*total_pop, 0.50*total_pop]
r_km = reach_thresholds(dist_np, pop_np, targets)
r_min = reach_thresholds(time_np, pop_np, targets)

acc = pd.DataFrame({
    "Comune_norm": master_names,
    "A_km_beta1_5": A_km,
    "A_time": A_t,
    "R25_km": r_km[:,0], "R50_km": r_km[:,1],
    "T25_min": r_min[:,0], "T50_min": r_min[:,1],
})

# Neighbor counts
neighbors = {"Comune_norm": master_names}
for thr in [10, 20, 30]:
    neighbors[f"neighbors_km_le_{thr}"] = np.sum((dist_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
for thr in [15, 30, 45]:
    neighbors[f"neighbors_min_le_{thr}"] = np.sum((time_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
neighbors_df = pd.DataFrame(neighbors)

# Merge all indicators
metrics = (ind[["Codice","Comune_norm","popolazione","in_Totale","out_Totale","saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc","indice_bilancio"]]
           .merge(acc, on="Comune_norm", how="left")
           .merge(neighbors_df, on="Comune_norm", how="left"))

# Provincial means (for diff table)
prov_means = metrics.drop(columns=["Codice","Comune_norm"]).mean(numeric_only=True)

# Quartiles for narrative
quartiles = metrics.drop(columns=["Codice","Comune_norm"]).quantile([0.25, 0.5, 0.75], numeric_only=True)
Q1 = quartiles.loc[0.25]; Q2 = quartiles.loc[0.5]; Q3 = quartiles.loc[0.75]

# ---------- Similarity (nearest neighbors) ----------
# Standardize indicators, set direction so that "higher-is-better" for all features used in similarity
sim_cols = [
    ("saldo_netto", +1.0),
    ("intensita_pendolarismo", +1.0),
    ("autocontenimento_pc", +1.0),
    ("attrattivita_pc", +1.0),
    ("A_km_beta1_5", +1.0),
    ("R50_km", -1.0),   # lower is better -> invert
    ("T50_min", -1.0),  # lower is better -> invert
    ("neighbors_km_le_20", +1.0),
    ("neighbors_min_le_30", +1.0),
]

X = metrics[[c for c,_ in sim_cols]].copy()
# Invert lower-is-better columns by multiplying by -1, then z-score
for c, w in sim_cols:
    if w < 0:
        X[c] = -X[c]
Xz = (X - X.mean())/X.std(ddof=0)

# Compute nearest neighbors (Euclidean in z-space)
from sklearn.metrics.pairwise import pairwise_distances
D = pairwise_distances(Xz.fillna(0.0).to_numpy(), metric="euclidean")
# For each i, get the 5 nearest j != i
neighbors_idx = {}
n = len(metrics)
for i in range(n):
    order = np.argsort(D[i])
    # first element is itself; pick next 5
    sim_ids = [int(j) for j in order[1:6]]
    neighbors_idx[i] = sim_ids

# Map index -> name and anchor
names = metrics["Comune_norm"].tolist()
anchors = {name: slugify(name) for name in names}

# ---------- Natural language description generator ----------
def describe_row(row):
    pieces = []
    name = row["Comune_norm"]
    # Role by saldo_netto
    if row["saldo_netto"] > Q2["saldo_netto"]:
        pieces.append("a regional **employment hub** (positive net inflow)")
    elif row["saldo_netto"] < Q2["saldo_netto"]:
        pieces.append("a **residential-leaning** municipality (net outflow)")
    else:
        pieces.append("**balanced** in job inflows and outflows")
    # Attractiveness & self-containment
    att_label = quantile_label(row["attrattivita_pc"], Q1["attrattivita_pc"], Q2["attrattivita_pc"], Q3["attrattivita_pc"], True)
    sc_label  = quantile_label(row["autocontenimento_pc"], Q1["autocontenimento_pc"], Q2["autocontenimento_pc"], Q3["autocontenimento_pc"], True)
    pieces.append(f"with **workplace attractiveness** {att_label} and **self-containment** {sc_label}")
    # Accessibility
    r50_label = quantile_label(row["R50_km"], Q1["R50_km"], Q2["R50_km"], Q3["R50_km"], False)
    t50_label = quantile_label(row["T50_min"], Q1["T50_min"], Q2["T50_min"], Q3["T50_min"], False)
    pieces.append(f"and **accessibility** (R50 km / T50 min) rated {r50_label} / {t50_label}.")
    desc = f"The municipality of **{name}** is {', '.join(pieces)}"
    return desc

# ---------- Build Markdown ----------
lines = []

# Title + TOC
lines.append("# Municipality Mobility Profiles — Trentino 2021")
lines.append("")
lines.append("## Index")
# sorted by name
for name in sorted(names):
    lines.append(f"- [{name}](#{anchors[name]})")
lines.append("")
lines.append("---")
lines.append("")

# Indicator set for the table (value + diff to provincial mean)
table_cols = [
    ("popolazione", "Population"),
    ("in_Totale", "Inbound (in_Totale)"),
    ("out_Totale", "Outbound (out_Totale)"),
    ("saldo_netto", "Net balance"),
    ("intensita_pendolarismo", "Commuting intensity"),
    ("autocontenimento_pc", "Self-containment pc"),
    ("attrattivita_pc", "Attractiveness pc"),
    ("indice_bilancio", "Balance index (in/out)"),
    ("A_km_beta1_5", "Accessibility potential (km, β=1.5)"),
    ("R50_km", "R50 (km)"),
    ("T50_min", "T50 (min)"),
    ("neighbors_km_le_20", "Neighbors ≤20 km"),
    ("neighbors_min_le_30", "Neighbors ≤30 min"),
]

# Build sections
for i, row in metrics.sort_values("Comune_norm").reset_index(drop=True).iterrows():
    name = row["Comune_norm"]
    anc = anchors[name]
    lines.append(f'<a id="{anc}"></a>')
    lines.append(f"## {name}")
    lines.append("")
    # Description
    lines.append(describe_row(row))
    lines.append("")
    # Table header
    lines.append("| Indicator | Value | Provincial mean | Δ (Value − Mean) |")
    lines.append("|---|---:|---:|---:|")
    for col, label in table_cols:
        val = row[col]
        mean = prov_means[col] if col in prov_means.index else np.nan
        diff = val - mean if pd.notna(val) and pd.notna(mean) else np.nan
        lines.append(f"| {label} | {fmt_num(val)} | {fmt_num(mean)} | {fmt_num(diff)} |")
    lines.append("")
    # Similar municipalities
    sims = neighbors_idx[i]
    sim_links = [f"[{metrics.iloc[j]['Comune_norm']}](#{anchors[metrics.iloc[j]['Comune_norm']]})" for j in sims]
    lines.append(f"**Similar municipalities:** {', '.join(sim_links)}")
    lines.append("")
    lines.append("---")
    lines.append("")

# Write file
with open(OUT_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

# Show preview (first ~200 lines) and provide link
preview_lines = "\n".join(lines[:200])
print("Generated:", OUT_MD)
print("\n--- PREVIEW (first 200 lines) ---\n")
print(preview_lines)

# Also show a compact DataFrame of means used
means_df = prov_means.reset_index()
means_df.columns = ["indicator","provincial_mean"]
print("Provincial means used in the profile tables")
means_df


Generated: documentation/profili_comuni.md

--- PREVIEW (first 200 lines) ---

# Municipality Mobility Profiles — Trentino 2021

## Index
- [Ala](#ala)
- [Albiano](#albiano)
- [Aldeno](#aldeno)
- [Altavalle](#altavalle)
- [Altopiano della Vigolana](#altopiano-della-vigolana)
- [Amblar-Don](#amblar-don)
- [Andalo](#andalo)
- [Arco](#arco)
- [Avio](#avio)
- [Baselga di Pinè](#baselga-di-pine)
- [Bedollo](#bedollo)
- [Besenello](#besenello)
- [Bieno](#bieno)
- [Bleggio Superiore](#bleggio-superiore)
- [Bocenago](#bocenago)
- [Bondone](#bondone)
- [Borgo Chiese](#borgo-chiese)
- [Borgo Lares](#borgo-lares)
- [Borgo Valsugana](#borgo-valsugana)
- [Borgo d'Anaunia](#borgo-danaunia)
- [Brentonico](#brentonico)
- [Bresimo](#bresimo)
- [Caderzone Terme](#caderzone-terme)
- [Calceranica al Lago](#calceranica-al-lago)
- [Caldes](#caldes)
- [Caldonazzo](#caldonazzo)
- [Calliano](#calliano)
- [Campitello di Fassa / Ciampedel](#campitello-di-fassa--ciampedel)
- [Campodenno](#campodenno)
- [Canal San

/tmp/ipykernel_54667/3659011968.py:94: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  time_min = time_hm.applymap(hmm_to_minutes)


,indicator,provincial_mean
0,popolazione,3262.632530
1,in_Totale,29.807229
2,out_Totale,44.234940
3,saldo_netto,-14.427711
4,intensita_pendolarismo,0.021741
5,autocontenimento_pc,0.107512
6,attrattivita_pc,0.007173
7,indice_bilancio,0.802263
8,A_km_beta1_5,3566.948372
9,A_time,14575.393354


In [ ]:
# Genera le schede dei comuni in italiano con introduzione iniziale
import unicodedata
from sklearn.metrics.pairwise import pairwise_distances


OUT_MD = DOCS / "profili_comuni.md"

# --- funzioni di supporto ---
def read_csv_auto(path):
    try: return pd.read_csv(path)
    except Exception: return pd.read_csv(path, sep=";")

def slugify(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii","ignore").decode("ascii")
    text = text.lower().strip().replace(" ", "-")
    text = "".join(ch for ch in text if ch.isalnum() or ch in "-_")
    return text

def fmt_num(x, digits=3):
    if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))):
        return "—"
    try:
        xf = float(x)
        if abs(xf) >= 1000 and float(xf).is_integer():
            return f"{int(xf):,}".replace(",", ".")
        if abs(xf) >= 100:
            return f"{xf:.1f}"
        return f"{xf:.{digits}f}"
    except Exception:
        return str(x)

def quantile_label_it(val, q1, q2, q3, maggiore_meglio=True):
    if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
        return "n.d."
    if maggiore_meglio:
        if val < q1: return "basso (Q1)"
        if val < q2: return "medio-basso (Q2)"
        if val < q3: return "medio-alto (Q3)"
        return "alto (Q4)"
    else:
        if val < q1: return "alto (Q4)"
        if val < q2: return "medio-alto (Q3)"
        if val < q3: return "medio-basso (Q2)"
        return "basso (Q1)"

# --- caricamento dei dati ---
municipal = read_csv_auto(DATA/"municipalities_trentino.csv")
incoming  = read_csv_auto(DATA/"incoming_movements_2021.csv")
outcoming = read_csv_auto(DATA/"outcoming_movements_2021.csv")
pop       = read_csv_auto(DATA/"trentino_population_2021.csv")
dist_km   = read_csv_auto(DATA/"distance_km_matrix.csv")
time_hm   = read_csv_auto(DATA/"time_hhmm_matrix.csv")

municipal["comune_norm"] = municipal["comune"].astype(str).str.strip()
municipal = municipal.sort_values("comune_norm", kind="stable")
master_names = municipal["comune_norm"].tolist()

def ensure_matrix(df, master_names, index_col_name="Unnamed: 0"):
    if index_col_name in df.columns:
        df = df.set_index(index_col_name)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    return df.loc[master_names, master_names]

dist_km = ensure_matrix(dist_km, master_names)
time_hm = ensure_matrix(time_hm, master_names)

def hmm_to_minutes(s):
    s = str(s)
    if ":" not in s: return np.nan
    h, m = s.split(":")[:2]
    return int(h)*60 + int(m)

time_min = time_hm.applymap(hmm_to_minutes)

# --- indicatori di base ---
for df in (incoming, outcoming):
    df["Comune_norm"] = df["Comune"].astype(str).str.strip()
    df["Codice"] = df["Codice"].astype(int)
pop["Codice"] = pop["Codice"].astype(int)

incoming_min = incoming[["Codice","Comune_norm","Stesso","Altro","Lordo","Netto","Totale"]].rename(columns={
    "Stesso":"in_Stesso","Altro":"in_Altro","Lordo":"in_Lordo","Netto":"in_Netto","Totale":"in_Totale"
})
outcoming_min = outcoming[["Codice","Comune_norm","Stesso","Altro","Lordo","Netto","Totale"]].rename(columns={
    "Totale":"out_Totale"
})

ind = (outcoming_min
       .merge(incoming_min, on="Codice", how="left", suffixes=("_out","_in"))
       .merge(pop[["Codice","popolazione"]], on="Codice", how="left"))
ind = ind.rename(columns={"Comune_norm_out":"Comune_norm"})

ind["saldo_netto"] = ind["in_Totale"] - ind["out_Totale"]
ind["intensita_pendolarismo"] = (ind["in_Totale"] + ind["out_Totale"]) / ind["popolazione"]
ind["autocontenimento_pc"] = ind["Stesso"] / ind["popolazione"]
ind["attrattivita_pc"] = ind["in_Totale"] / ind["popolazione"]
ind["indice_bilancio"] = np.where(ind["out_Totale"]>0, ind["in_Totale"]/ind["out_Totale"], np.nan)

# --- Accessibilità ---
pop_vec = (pop.rename(columns={"Codice":"id"})
              .merge(municipal[["id","comune_norm"]], on="id", how="right")
              .set_index("comune_norm")["popolazione"]
              .astype(float))
beta = 1.5
dist_np = dist_km.to_numpy().astype(float)
time_np = time_min.to_numpy().astype(float)
pop_np = pop_vec.reindex(master_names).to_numpy().astype(float)

dist_np_z = dist_np.copy(); np.fill_diagonal(dist_np_z, np.nan)
time_np_z = time_np.copy(); np.fill_diagonal(time_np_z, np.nan)
with np.errstate(divide='ignore', invalid='ignore'):
    A_km = np.nansum(pop_np / np.power(dist_np_z, beta), axis=1)
    A_t  = np.nansum(pop_np / (1.0 + time_np_z), axis=1)

def reach_thresholds(matrix, pop_np, targets):
    n = matrix.shape[0]
    out = np.full((n, len(targets)), np.nan)
    for i in range(n):
        vals = matrix[i, :].copy(); vals[i] = np.nan
        order = np.argsort(vals)
        sorted_d, sorted_p = vals[order], pop_np[order]
        cum = np.cumsum(np.nan_to_num(sorted_p))
        for k, tgt in enumerate(targets):
            idx = np.searchsorted(cum, tgt, side="left")
            out[i, k] = sorted_d[idx] if idx < len(sorted_d) else np.nan
    return out

total_pop = np.nansum(pop_np)
targets = [0.25*total_pop, 0.50*total_pop]
r_km = reach_thresholds(dist_np, pop_np, targets)
r_min = reach_thresholds(time_np, pop_np, targets)

acc = pd.DataFrame({
    "Comune_norm": master_names,
    "A_km_beta1_5": A_km,
    "A_time": A_t,
    "R25_km": r_km[:,0], "R50_km": r_km[:,1],
    "T25_min": r_min[:,0], "T50_min": r_min[:,1],
})

neighbors = {"Comune_norm": master_names}
for thr in [10, 20, 30]:
    neighbors[f"neighbors_km_le_{thr}"] = np.sum((dist_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
for thr in [15, 30, 45]:
    neighbors[f"neighbors_min_le_{thr}"] = np.sum((time_np <= thr) & (~np.eye(len(master_names), dtype=bool)), axis=1)
neighbors_df = pd.DataFrame(neighbors)

metrics = (ind[["Codice","Comune_norm","popolazione","in_Totale","out_Totale","saldo_netto","intensita_pendolarismo","autocontenimento_pc","attrattivita_pc","indice_bilancio"]]
           .merge(acc, on="Comune_norm", how="left")
           .merge(neighbors_df, on="Comune_norm", how="left"))

prov_means = metrics.drop(columns=["Codice","Comune_norm"]).mean(numeric_only=True)
quartiles = metrics.drop(columns=["Codice","Comune_norm"]).quantile([0.25, 0.5, 0.75], numeric_only=True)
Q1, Q2, Q3 = quartiles.loc[0.25], quartiles.loc[0.5], quartiles.loc[0.75]

# --- Similarità
sim_cols = [
    ("saldo_netto", +1.0), ("intensita_pendolarismo", +1.0),
    ("autocontenimento_pc", +1.0), ("attrattivita_pc", +1.0),
    ("A_km_beta1_5", +1.0), ("R50_km", -1.0), ("T50_min", -1.0),
    ("neighbors_km_le_20", +1.0), ("neighbors_min_le_30", +1.0),
]
X = metrics[[c for c,_ in sim_cols]].copy()
for c, w in sim_cols:
    if w < 0: X[c] = -X[c]
Xz = (X - X.mean())/X.std(ddof=0)
D = pairwise_distances(Xz.fillna(0.0).to_numpy(), metric="euclidean")
neighbors_idx = {i: [int(j) for j in np.argsort(D[i])[1:6]] for i in range(len(metrics))}
names = metrics["Comune_norm"].tolist()
anchors = {n: slugify(n) for n in names}

# --- descrizione testuale
def descrivi(row):
    frasi = []
    if row["saldo_netto"] > Q2["saldo_netto"]:
        frasi.append("un **polo occupazionale** (saldo positivo dei flussi di lavoro)")
    elif row["saldo_netto"] < Q2["saldo_netto"]:
        frasi.append("un comune a prevalenza **residenziale** (più lavoratori in uscita che in entrata)")
    else:
        frasi.append("un comune **bilanciato** nei flussi in entrata e in uscita")
    att = quantile_label_it(row["attrattivita_pc"], Q1["attrattivita_pc"], Q2["attrattivita_pc"], Q3["attrattivita_pc"], True)
    sc  = quantile_label_it(row["autocontenimento_pc"], Q1["autocontenimento_pc"], Q2["autocontenimento_pc"], Q3["autocontenimento_pc"], True)
    r50 = quantile_label_it(row["R50_km"], Q1["R50_km"], Q2["R50_km"], Q3["R50_km"], False)
    t50 = quantile_label_it(row["T50_min"], Q1["T50_min"], Q2["T50_min"], Q3["T50_min"], False)
    frasi.append(f"con un livello di **attrattività lavorativa** {att} e **autocontenimento** {sc}")
    frasi.append(f"e un’**accessibilità** (R50 km / T50 min) classificata come {r50} / {t50}.")
    return f"Il comune di **{row['Comune_norm']}** si caratterizza come {', '.join(frasi)}"

# --- generazione markdown
lines = [
"# 🗺️ Schede di Mobilità dei Comuni del Trentino — 2021",
"",
"Le seguenti schede descrivono il **profilo di mobilità** di ciascun comune del Trentino, basato sui microdati dei movimenti pendolari 2021.",
"Ogni scheda riporta un commento sintetico in linguaggio naturale, una tabella di indicatori confrontata con la **media provinciale**, e una lista di **comuni simili** calcolata tramite analisi di similarità sugli indicatori principali.",
"Questo documento fa parte del progetto **tracetn — Trentino Commuter Analysis**.",
"",
"## Indice"
]
for n in sorted(names):
    lines.append(f"- [{n}](#{anchors[n]})")
lines.append("\n---\n")

colonne = [
    ("popolazione","Popolazione"),
    ("in_Totale","Lavoratori in entrata"),
    ("out_Totale","Lavoratori in uscita"),
    ("saldo_netto","Saldo netto"),
    ("intensita_pendolarismo","Intensità pendolarismo"),
    ("autocontenimento_pc","Autocontenimento (%)"),
    ("attrattivita_pc","Attrattività (%)"),
    ("indice_bilancio","Indice di bilancio (in/out)"),
    ("A_km_beta1_5","Accessibilità potenziale (km, β=1.5)"),
    ("R50_km","R50 (km)"),
    ("T50_min","T50 (min)"),
    ("neighbors_km_le_20","Comuni entro 20 km"),
    ("neighbors_min_le_30","Comuni entro 30 min"),
]

for i, row in metrics.sort_values("Comune_norm").reset_index(drop=True).iterrows():
    n = row["Comune_norm"]; anc = anchors[n]
    lines.append(f'<a id="{anc}"></a>\n## {n}\n')
    lines.append(descrivi(row) + "\n")
    lines.append("| Indicatore | Valore | Media provinciale | Δ (Valore − Media) |")
    lines.append("|---|---:|---:|---:|")
    for c, label in colonne:
        val, mean = row[c], prov_means.get(c, np.nan)
        diff = (val - mean) if pd.notna(val) and pd.notna(mean) else np.nan
        lines.append(f"| {label} | {fmt_num(val)} | {fmt_num(mean)} | {fmt_num(diff)} |")
    sims = neighbors_idx[i]
    link_sim = [f"[{metrics.iloc[j]['Comune_norm']}](#{anchors[metrics.iloc[j]['Comune_norm']]})" for j in sims]
    lines.append("\n**Comuni simili:** " + ", ".join(link_sim) + "\n---\n")

with open(OUT_MD,"w",encoding="utf-8") as f:
    f.write("\n".join(lines))

print("✅ File generato:", OUT_MD)
print("Numero righe:", len(lines))
print("\nAnteprima:\n", "\n".join(lines[:40]))


✅ File generato: documentation/municipality_profiles.md
Numero righe: 3162

Anteprima:
 # 🗺️ Schede di Mobilità dei Comuni del Trentino — 2021

Le seguenti schede descrivono il **profilo di mobilità** di ciascun comune del Trentino, basato sui microdati dei movimenti pendolari 2021.
Ogni scheda riporta un commento sintetico in linguaggio naturale, una tabella di indicatori confrontata con la **media provinciale**, e una lista di **comuni simili** calcolata tramite analisi di similarità sugli indicatori principali.
Questo documento fa parte del progetto **tracetn — Trentino Commuter Analysis**.

## Indice
- [Ala](#ala)
- [Albiano](#albiano)
- [Aldeno](#aldeno)
- [Altavalle](#altavalle)
- [Altopiano della Vigolana](#altopiano-della-vigolana)
- [Amblar-Don](#amblar-don)
- [Andalo](#andalo)
- [Arco](#arco)
- [Avio](#avio)
- [Baselga di Pinè](#baselga-di-pine)
- [Bedollo](#bedollo)
- [Besenello](#besenello)
- [Bieno](#bieno)
- [Bleggio Superiore](#bleggio-superiore)
- [Bocenago](#bocenago)


/tmp/ipykernel_54667/45026273.py:74: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  time_min = time_hm.applymap(hmm_to_minutes)
